# Feature selection — SelectKBest vs RFE

One small experiment a day. Seed fixed for reproducibility.

Breast cancer has 30 features. Do we need all of them? Compare univariate selection vs recursive elimination at k=10.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor,
                              HistGradientBoostingClassifier, VotingClassifier,
                              IsolationForest)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix,
                             classification_report, mean_squared_error,
                             mean_absolute_error, r2_score, silhouette_score)

from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, RFE, f_classif


## Comparison

In [ ]:
SEED = 1108
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y)
base = LogisticRegression(max_iter=5000)
sels = {"all": None,
        "kbest": SelectKBest(f_classif, k=10),
        "rfe": RFE(base, n_features_to_select=10)}
for name, sel in sels.items():
    if sel is None:
        Xt_tr, Xt_te, feats = X_train, X_test, list(X.columns)
    else:
        Xt_tr = sel.fit_transform(X_train, y_train)
        Xt_te = sel.transform(X_test)
        feats = list(X.columns[sel.get_support()])
    m = LogisticRegression(max_iter=5000).fit(Xt_tr, y_train)
    print(f"{name:>5}: accuracy {m.score(Xt_te, y_test):.4f} | features {len(feats)}")
    if name == "rfe":
        print("   rfe picks:", feats)


  all: accuracy 0.9357 | features 30
kbest: accuracy 0.9298 | features 10
  rfe: accuracy 0.9357 | features 10
   rfe picks: ['mean radius', 'mean concavity', 'texture error', 'perimeter error', 'worst radius', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry']


## Takeaways

- 10 features ≈ all 30 here — heavy redundancy in the measurements.
- RFE and KBest mostly agree; the disagreement is in correlated pairs.